In [1]:
import pandas as pd
import glob
import os
import re

In [ ]:
input_folder = "C:/Users/rashe/Desktop/Cattle Scan/Farm Data/Farm-id-68"
output_folder = "Cleaned_Data_files"

os.makedirs(output_folder, exist_ok=True)

In [ ]:
def clean_data(file_path):
 
    match = re.search(r'animalId-(\d+)', os.path.basename(file_path))
    calf_id = match.group(1) if match else None

   
    df = pd.read_csv(file_path)

  
    df = df.drop(columns=['waterTemp', 'lactationNumber'], errors='ignore')
    df.rename(columns={'date': 'datetime'}, inplace=True)

    df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')
    df['date'] = df['datetime'].dt.date
    df['time'] = df['datetime'].dt.time
    df['month'] = df['datetime'].dt.month

    for col in ['airTemp', 'humidity', 'accel']:
        if col in df.columns:
            df[col] = df[col].interpolate(method='linear')

    if 'rumination' in df.columns:
        df['rumination'] = df['rumination'].interpolate(method='nearest')
        df['rumination'] = df['rumination'].fillna(0)
        df['rumination'] = df['rumination'].round().astype(int)

    if 'aveHerdTemp' in df.columns:
        df['aveHerdTemp'] = df['aveHerdTemp'].ffill()

    # Add calf ID column
    df['id'] = calf_id

    
    output_name = f"cleaned-{os.path.basename(file_path)}"
    output_path = os.path.join(output_folder, output_name)
    df.to_csv(output_path, index=False)

    print(f"✅ Cleaned and saved: {output_name}")


In [ ]:
for file in glob.glob(f"{input_folder}/*.csv"):
    clean_data(file)

print("All datasets cleaned successfully!")